## Import libraries

In [1]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
from datasets import load_dataset, DatasetDict
import numpy as np
import evaluate
import torch
import os
import shutil

## Prepare the data

### Get the data

In [3]:
ds = load_dataset("imdb", split=["train", "test"])

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [4]:
test_val_split = ds[1].train_test_split(test_size=0.5, seed=42)

# Create the final DatasetDict with train, new test, and validation splits
ds = DatasetDict({
    "train": ds[0],
    "test": test_val_split["test"],
    "validation": test_val_split["train"] # The 'train' part of the split becomes the validation set
})

In [5]:
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 12500
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 12500
    })
})

In [6]:
# sample

ds["train"][-1]["text"], ds["train"][0]["label"]

('The story centers around Barry McKenzie who must go to England if he wishes to claim his inheritance. Being about the grossest Aussie shearer ever to set foot outside this great Nation of ours there is something of a culture clash and much fun and games ensue. The songs of Barry McKenzie(Barry Crocker) are highlights.',
 0)

## Define tokenizer, collator, model

In [7]:
checkpoint = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(checkpoint, return_tensors="pt")
# Set the padding token for the tokenizer
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
   checkpoint,
   pad_token_id=tokenizer.eos_token_id,
   num_labels=2
)
collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key                  | Status     | 
---------------------+------------+-
h.{0...11}.attn.bias | UNEXPECTED | 
score.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
tokenized_fn = lambda x: tokenizer(x["text"], truncation=True)

tokenized_ds = ds.map(tokenized_fn, batched=True, remove_columns=["text"])
tokenized_ds = tokenized_ds.rename_column("label", "labels")

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

Map:   0%|          | 0/12500 [00:00<?, ? examples/s]

## Train the model

In [9]:
train_args = TrainingArguments(
  output_dir="model_log",
  num_train_epochs=3,
  learning_rate=8e-6,
  per_device_train_batch_size=6,
  logging_steps=300,
  max_grad_norm=3,
  gradient_accumulation_steps=2,
  fp16=True
)

In [10]:
# Load metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    # 'binary' average for f1_score
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {"accuracy": accuracy["accuracy"], "f1_score": f1["f1"]}

In [11]:
trainer = Trainer(
  model,
  args=train_args,
  train_dataset=tokenized_ds["train"],
  eval_dataset=tokenized_ds["validation"],
  data_collator=collator,
  compute_metrics=compute_metrics
)

In [12]:
trainer.train()
print("Trained model successfully!")

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
300,2.065979
600,1.004911
900,0.886712
1200,0.821813
1500,0.765434
1800,0.749125
2100,0.725807
2400,0.636511
2700,0.640582
3000,0.638090


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Trained model successfully!


## Final performance check

In [13]:
trainer.evaluate(tokenized_ds["test"])

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{'eval_loss': 0.3425486981868744,
 'eval_accuracy': 0.94168,
 'eval_f1_score': 0.941346850108617,
 'eval_runtime': 392.3754,
 'eval_samples_per_second': 31.857,
 'eval_steps_per_second': 1.993,
 'epoch': 3.0}

In [14]:
@torch.no_grad()
def predict(text, model):
  # Tokenize the input text and ensure it returns PyTorch tensors
  device = next(model.parameters()).device # Get the device of the first model parameter
  tokenized_input = tokenizer(text, truncation=True, padding=True, return_tensors="pt").to(device)

  logits = model(**tokenized_input).logits
  prob = torch.softmax(logits, dim=-1)
  pred = torch.argmax(prob, dim=-1).item()

  # Get the probability of the predicted class
  predicted_prob = prob[0, pred].item()

  # Map the class ID to a meaningful label (0: Negative, 1: Positive)
  label = "Positive" if (pred == 1) else "Negative"

  return label, predicted_prob

In [15]:
text1 = "This movie was absolutely fantastic! The acting was superb, the plot was engaging, and I was on the edge of my seat the entire time. A true masterpiece!"

print(predict(text1, model))

text2 = "This movie was a complete disaster. The acting was terrible, the plot was nonsensical, and I was bored from beginning to end. A waste of time!"

print(predict(text2, model))

text3 = "While the visuals were stunning, the storyline was disjointed and the characters lacked depth, making it a visually appealing but ultimately unfulfilling experience."
print(predict(text3, model))

('Positive', 0.9981065988540649)
('Negative', 0.9997033476829529)
('Negative', 0.9900566935539246)


In [16]:
# Remove the log directory
if os.path.exists("model_log"):
    shutil.rmtree("model_log")
    print("Removed model_log directory.")

# Save the final model
model_save_path = "./final_model"
trainer.save_model(model_save_path)
print(f"Final model saved to {model_save_path}")

Removed model_log directory.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Final model saved to ./final_model


In [17]:
archive_name = "final_model_archive"

# Create a zip archive of the directory
shutil.make_archive(archive_name, 'zip', model_save_path)
print(f"Directory '{model_save_path}' successfully zipped to '{archive_name}.zip'.")

# Remove the original directory
if os.path.exists(model_save_path):
    shutil.rmtree(model_save_path)
    print(f"Original directory '{model_save_path}' removed.")

Directory './final_model' successfully zipped to 'final_model_archive.zip'.
Original directory './final_model' removed.
